## Create an Gen AI App using Langchain

In [2]:
## Data Ingestion - Web Scraping
from langchain_community.document_loaders import WebBaseLoader
## Load all API keys and configuration from .env file
import os
from dotenv import load_dotenv
# load environment variables from a .env file into your Python application's environment
load_dotenv() 
# Sets some variables into Python application's environment.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# LangSmith Tracing configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4o")
print(llm)

profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001C3EE804090> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C3EECF5890> root_client=<openai.OpenAI object at 0x000001C3EE759B90> root_async_client=<openai.AsyncOpenAI object at 0x000001C3EECF5390> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [4]:
## Data Ingestion - Scrape data from website (Load data from URL)
website_loader = WebBaseLoader("https://docs.langchain.com/langsmith/data-storage-and-privacy")
## Load the data from the website (data from url converted into document format)
document = website_loader.load()
document

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/data-storage-and-privacy', 'title': 'Data storage and privacy - Docs by LangChain', 'language': 'en'}, page_content='Data storage and privacy - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationData managementData storage and privacyGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewPlansCreate an account and API keyAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementData storage and privacyData purging for complianceAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusOn this pageCLIAgent ServerLangSmith tracingIn-memory development serverStandalone ServerStudioQuick referenceAdditional resourcesData man

In [5]:
## Data Transformation - Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
split_docs = text_splitter.split_documents(document)
split_docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/data-storage-and-privacy', 'title': 'Data storage and privacy - Docs by LangChain', 'language': 'en'}, page_content='Data storage and privacy - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationData managementData storage and privacyGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewPlansCreate an account and API keyAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementData storage and privacyData purging for complianceAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusOn this pageCLIAgent ServerLangSmith tracingIn-memory development serverStandalone ServerStudioQuick referenceAdditional resourcesData man

In [6]:
## Embedding - OpenAI Embeddings
from langchain_openai import OpenAIEmbeddings
embedding_handle = OpenAIEmbeddings(model="text-embedding-3-large") 

In [7]:
## Store embeddings in FAISS vector store
from langchain_community.vectorstores import FAISS
vector_store_db = FAISS.from_documents(split_docs, embedding_handle)
vector_store_db

In [8]:
## Query from the vector store
query = "The Agent Server provides a durable execution runtime that relies on persisting"
## get similar documents from the vector store
response = vector_store_db.similarity_search(query)
response[0].page_content

'\u200bAgent Server\nThe Agent Server provides a durable execution runtime that relies on persisting checkpoints of your application state, long-term memories, thread metadata, assistants, and similar resources to the local file system or a database. Unless you have deliberately customized the storage location, this information is either written to local disk (for langgraph dev) or a PostgreSQL database (for langgraph up and in all deployments).\n\u200bLangSmith tracing\nWhen running the Agent server (either in-memory or in Docker), LangSmith tracing may be enabled to facilitate faster debugging and offer observability of graph state and LLM prompts in production. You can always disable tracing by setting LANGSMITH_TRACING=false in your server’s runtime environment.'

In [27]:
## Retrieval chain, document chains
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template(
    """
    Answewr the question based on the context below.
    <context>
    {context}
    </context>
    Question: {input}
    Answer:
    """
)
## provide a context to prompt template regarding any input
document_chain = prompt | llm | StrOutputParser()
document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answewr the question based on the context below.\n    <context>\n    {context}\n    </context>\n    Question: {input}\n    Answer:\n    '), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C3EE804090>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions 

In [28]:
## Create retriever chain
from langchain_core.runnables import RunnablePassthrough
## Convert vector store to retriever
retriever = vector_store_db.as_retriever()
# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [31]:
retriever_chain = (
    {
        "context": retriever | format_docs,  # Add format_docs here
        "input": RunnablePassthrough()
    } 
    | document_chain
)

In [34]:
## Get the response from llm
response = retriever_chain.invoke(
    "What does the Agent Server rely on for durable execution?"
)
response

'The Agent Server relies on persisting checkpoints of your application state, long-term memories, thread metadata, assistants, and similar resources to the local file system or a database for durable execution.'